## Enriquecimento com Dados de CNPJ

Para cada CNPJ de fornecedor vencedor encontrado em `propostas_pe.csv` (produzido pelo NB02b),
consulta dados cadastrais da empresa via BrasilAPI.
Extrai: porte, natureza juridica, data de abertura, municipio/UF, CNAE principal.

Fontes:
- `data/raw/propostas_pe.csv` — CNPJ dos fornecedores vencedores (`cnpj_fornecedor`)
- BrasilAPI: `https://brasilapi.com.br/api/cnpj/v1/{cnpj}`

In [4]:
import urllib.request
import json
import pandas as pd
import time
import os

os.makedirs("data/raw", exist_ok=True)
BRASILAPI_BASE = "https://brasilapi.com.br/api/cnpj/v1"
print("Setup OK")

Setup OK


### 1. Carrega propostas e extrai CNPJs unicos

In [5]:
propostas = pd.read_csv("data/raw/propostas_pe.csv", dtype=str)
print(f"{len(propostas)} propostas carregadas")
print("Colunas:", list(propostas.columns))

# Identifica coluna de CNPJ do fornecedor vencedor
# NB02b produz a coluna 'cnpj_fornecedor'
cnpj_col = next(
    (c for c in propostas.columns if "cnpj" in c.lower() and "orgao" not in c.lower()),
    None
)
print(f"\nColuna de CNPJ do fornecedor detectada: {cnpj_col}")
if cnpj_col:
    cnpjs_unicos = propostas[cnpj_col].dropna().unique()
    print(f"{len(cnpjs_unicos)} CNPJs únicos de fornecedores vencedores")

75502 propostas carregadas
Colunas: ['idCompra', 'orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'descricaoItem', 'cnpj_fornecedor', 'nome_fornecedor', 'valor_total_homologado', 'situacao_resultado']

Coluna de CNPJ do fornecedor detectada: cnpj_fornecedor
9458 CNPJs únicos de fornecedores vencedores


### 2. Debug - Inspecionar estrutura da BrasilAPI

In [6]:
# Testa com um CNPJ de amostra
cnpj_teste = cnpjs_unicos[0].replace(".", "").replace("/", "").replace("-", "").zfill(14)
url_teste = f"{BRASILAPI_BASE}/{cnpj_teste}"
print(f"Consultando: {url_teste}")

try:
    req = urllib.request.Request(url_teste, headers={"User-Agent": "tcc-research/1.0"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read().decode())
    print("\nCampos disponiveis:")
    for k, v in data.items():
        print(f"  {k}: {str(v)[:100]}")
except Exception as e:
    print(f"Erro: {e}")

Consultando: https://brasilapi.com.br/api/cnpj/v1/40230669000194

Campos disponiveis:
  uf: PE
  cep: 55150290
  qsa: []
  cnpj: 40230669000194
  pais: None
  email: None
  porte: MICRO EMPRESA
  bairro: CENTRO
  numero: 
  ddd_fax: 
  municipio: BELO JARDIM
  logradouro: 
  cnae_fiscal: 8230001
  codigo_pais: None
  complemento: 
  codigo_porte: 1
  razao_social: 40.230.669 EMERSON PAULO SILVA FARIAS
  nome_fantasia: 
  capital_social: 1000
  ddd_telefone_1: 
  ddd_telefone_2: 
  opcao_pelo_mei: True
  codigo_municipio: 2333
  cnaes_secundarios: [{'codigo': 7729202, 'descricao': 'Aluguel de móveis, utensílios e aparelhos de uso doméstico e pess
  natureza_juridica: Empresário (Individual)
  regime_tributario: []
  situacao_especial: 
  opcao_pelo_simples: True
  situacao_cadastral: 2
  data_opcao_pelo_mei: 2021-01-02
  data_exclusao_do_mei: None
  cnae_fiscal_descricao: Serviços de organização de feiras, congressos, exposições e festas
  codigo_municipio_ibge: 2601706
  data_inicio_at

### 3. Consulta CNPJ para todos os fornecedores

Rate limit da BrasilAPI: ~3 req/s. Sleep de 0.4s entre chamadas.

In [7]:
CAMPOS_INTERESSE = [
    "cnpj",
    "razao_social",
    "nome_fantasia",
    "descricao_situacao_cadastral",  # ex: "ATIVA"
    "data_inicio_atividade",
    "natureza_juridica",             # ex: "Empresário (Individual)"
    "porte",                         # ex: "MICRO EMPRESA"
    "municipio",
    "uf",
    "codigo_municipio_ibge",         # útil para join geográfico alternativo
    "cnae_fiscal",
    "cnae_fiscal_descricao",
    "capital_social",
    "opcao_pelo_mei",                # bool — MEI indica porte muito pequeno
    "opcao_pelo_simples",            # bool — Simples Nacional
]

def consultar_cnpj(cnpj_raw):
    cnpj = str(cnpj_raw).replace(".", "").replace("/", "").replace("-", "").strip().zfill(14)
    try:
        req = urllib.request.Request(
            f"{BRASILAPI_BASE}/{cnpj}",
            headers={"User-Agent": "tcc-research/1.0"}
        )
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = json.loads(resp.read().decode())
        return {k: data.get(k) for k in CAMPOS_INTERESSE}
    except Exception:
        return {"cnpj": cnpj, "erro": True}

# Teste com os primeiros 20 CNPJs
resultados_cnpj = []
cnpjs_teste = cnpjs_unicos[:20]
for i, cnpj in enumerate(cnpjs_teste):
    res = consultar_cnpj(cnpj)
    resultados_cnpj.append(res)
    time.sleep(0.4)
    if (i + 1) % 5 == 0:
        print(f"  {i+1}/{len(cnpjs_teste)} consultados")

df_cnpj_teste = pd.DataFrame(resultados_cnpj)
print(f"\n{len(df_cnpj_teste)} CNPJs consultados")
print(df_cnpj_teste[["cnpj", "porte", "natureza_juridica", "municipio", "opcao_pelo_mei"]].head(5))

  5/20 consultados
  10/20 consultados
  15/20 consultados
  20/20 consultados

20 CNPJs consultados
             cnpj          porte              natureza_juridica  \
0  40230669000194  MICRO EMPRESA        Empresário (Individual)   
1  33040635000171         DEMAIS  Sociedade Empresária Limitada   
2  00010474760456            NaN                            NaN   
3  47270709000170  MICRO EMPRESA  Sociedade Empresária Limitada   
4  20852792000130  MICRO EMPRESA  Sociedade Empresária Limitada   

          municipio opcao_pelo_mei  
0       BELO JARDIM           True  
1    RIO DE JANEIRO           None  
2               NaN            NaN  
3  JARDIM DO SERIDO          False  
4    CAMPINA GRANDE          False  


In [8]:
checkpoint_file = "data/raw/fornecedores_cnpj.csv"

# Resume: pula CNPJs já processados
if os.path.exists(checkpoint_file):
    df_existente = pd.read_csv(checkpoint_file, dtype=str)
    cnpjs_prontos = set(df_existente["cnpj"].dropna().str.replace(r"\D", "", regex=True).str.zfill(14))
    resultados_cnpj = df_existente.to_dict("records")
    print(f"Retomando: {len(cnpjs_prontos)} CNPJs já processados, {len(resultados_cnpj)} registros")
else:
    cnpjs_prontos = set()
    resultados_cnpj = []

def normaliza(cnpj):
    return str(cnpj).replace(".", "").replace("/", "").replace("-", "").strip().zfill(14)

pendentes = [c for c in cnpjs_unicos if normaliza(c) not in cnpjs_prontos]
print(f"{len(pendentes)}/{len(cnpjs_unicos)} CNPJs a processar")

for i, cnpj in enumerate(pendentes):
    res = consultar_cnpj(cnpj)
    resultados_cnpj.append(res)
    time.sleep(0.4)
    if (i + 1) % 200 == 0:
        pd.DataFrame(resultados_cnpj).to_csv(checkpoint_file, index=False)
        print(f"  Checkpoint {i+1}/{len(pendentes)}")

fornecedores_cnpj = pd.DataFrame(resultados_cnpj)
fornecedores_cnpj.to_csv(checkpoint_file, index=False)
print(f"\n{len(fornecedores_cnpj)} fornecedores salvos em {checkpoint_file}")
erros = (fornecedores_cnpj.get("erro", pd.Series(dtype=object)) == True).sum()
print(f"Erros de consulta: {erros}")

9458/9458 CNPJs a processar
  Checkpoint 200/9458
  Checkpoint 400/9458
  Checkpoint 600/9458
  Checkpoint 800/9458
  Checkpoint 1000/9458
  Checkpoint 1200/9458
  Checkpoint 1400/9458
  Checkpoint 1600/9458
  Checkpoint 1800/9458
  Checkpoint 2000/9458
  Checkpoint 2200/9458
  Checkpoint 2400/9458
  Checkpoint 2600/9458
  Checkpoint 2800/9458
  Checkpoint 3000/9458
  Checkpoint 3200/9458
  Checkpoint 3400/9458
  Checkpoint 3600/9458
  Checkpoint 3800/9458
  Checkpoint 4000/9458
  Checkpoint 4200/9458
  Checkpoint 4400/9458
  Checkpoint 4600/9458
  Checkpoint 4800/9458
  Checkpoint 5000/9458
  Checkpoint 5200/9458
  Checkpoint 5400/9458
  Checkpoint 5600/9458
  Checkpoint 5800/9458
  Checkpoint 6000/9458
  Checkpoint 6200/9458
  Checkpoint 6400/9458
  Checkpoint 6600/9458
  Checkpoint 6800/9458
  Checkpoint 7000/9458
  Checkpoint 7200/9458
  Checkpoint 7400/9458
  Checkpoint 7600/9458
  Checkpoint 7800/9458
  Checkpoint 8000/9458
  Checkpoint 8200/9458
  Checkpoint 8400/9458
  Checkpoi